# US Used Vehicle Resales — Bad-Buy Prediction

**BINÄRE KLASSIFIKATION · FEHLKÄUFE BEI GEBRAUCHTWAGEN-AUKTIONEN VOR DEM KAUF ERKENNEN**

---

## Content

- [Szenario](#szenario)
- [Aufgabe](#aufgabe)
- [Ansatz](#ansatz)
- [Datenbasis](#datenbasis)
- [Navigation](#navigation)
- [Setup](#setup)

## Szenario

Ein US-Gebrauchtwagenhändler kauft Fahrzeuge günstig in Onlineauktionen ein, um sie auf der
eigenen Plattform gewinnbringend weiterzuverkaufen. Das grösste Risiko dabei: ein ersteigertes
Auto entpuppt sich als **"Bad Buy"** (ein "Montagsauto") — ein Fahrzeug mit schwerwiegenden,
herstellungsbedingten Mängeln, die Sicherheit, Nutzbarkeit oder Wert erheblich beeinträchtigen und
sich nicht wirtschaftlich reparieren lassen. Solche Wagen können nicht weiterverkauft werden und
verursachen neben den Anschaffungs- auch **Folgekosten** (Lagerung, Reparatur, Wertverlust).

Der Auftrag der Geschäftsführung: möglichst viele dieser Fehlkäufe **vor dem Kauf** ausschliessen
und so die Einkäufer bei der riesigen Menge an Auktionsangeboten entlasten — **ohne dabei zu viele
gute Käufe fälschlich auszusortieren**. Es geht also nicht um maximale Treffsicherheit, sondern um
einen brauchbaren **Precision/Recall-Kompromiss** auf einer seltenen Positivklasse.

> Herkunft: StackFuel-Abschlussprojekt (Modul 3, Kapitel 4). Die vollständige Original-Aufgabe →
> [`../docs/ASSIGNMENT.md`](../docs/ASSIGNMENT.md).

## Aufgabe

**Binäre Klassifikation** auf der Zielvariable `IsBadBuy`:

| Wert | Bedeutung |
|:----:|:----------|
| `0`  | kein Montagsauto — guter Kauf |
| `1`  | Montagsauto — Fehlkauf |

Die Klassen sind stark **unbalanciert** (nur ~12 % Bad Buys). Accuracy ist damit wertlos — ein
Modell, das immer "guter Kauf" sagt, wäre schon ~88 % "genau". Bewertet wird deshalb der
**F1-Score der Bad-Buy-Klasse** (bzw. Recall / Precision / PR-AUC).

**Zielmetrik der Prüfung:** ein **F1 > 0.40** auf dem verdeckten Zieldatensatz `features_aim.csv`,
dessen Labels nur die Prüfer kennen. Das Deliverable sind die Predictions für diesen Datensatz.

## Ansatz

Standard-DS-Workflow: Exploration → Aufbereitung → Modellierung → Evaluation, alle Schritte in
Funktionen/Pipelines gekapselt, damit sie am Ende deterministisch auf `features_aim` angewendet
werden können.

**Getestete Modelle** (systematischer Sweep über Feature-Sets × Modell-Familien, protokolliert mit
einem selbstgebauten `ModelTracker` — 448 Runs, → [`05_experiment_framework.ipynb`](05_experiment_framework.ipynb)):

| Familie | Varianten |
|:--------|:----------|
| Logistische Regression | Baseline (8 Features), Ridge (L2), **Lasso (L1)**, Elastic-Net — alle `class_weight='balanced'` |
| Random Forest | shallow, deep |
| Gradient Boosting | HistGradientBoosting (std, aggressive) |

**Champion: Logistische Regression mit L1-Penalty** auf dem grossen Feature-Set — höchster Recall
bei vertretbarer Precision. Der Entscheidungs-Threshold wird zusätzlich auf den F1 getunt. Die
finalen, vergleichbaren Testzahlen stehen in [`04_evaluation.ipynb`](04_evaluation.ipynb) (Single
Source of Truth).

## Datenbasis

| Datei | Inhalt |
|:------|:-------|
| `data/01_raw/data_train.csv` | Trainingsdaten mit Label `IsBadBuy` (33 Spalten, eine Zeile je Fahrzeug) |
| `data/01_raw/features_aim.csv` | Zieldatensatz **ohne** Label — hierfür werden Predictions erzeugt (Deliverable) |

Vollständige Spaltenbeschreibung → [`../DATA_DICTIONARY.md`](../DATA_DICTIONARY.md) ·
Original-Aufgabenstellung → [`../docs/ASSIGNMENT.md`](../docs/ASSIGNMENT.md).

> Rohdaten und Modelle sind via `.gitignore` aus dem Repo ausgeschlossen.

## Navigation

| # | Notebook | Inhalt |
|:--|:---------|:-------|
| 00 | `00_introduction.ipynb` | Dieser Einstieg |
| 01 | `01_exploring.ipynb` | Explorative Datenanalyse (EDA) |
| 02 | `02_processing.ipynb` | Cleaning, Feature Engineering |
| 03 | `03_modelling-prep.ipynb` | Modelling-Vorbereitung (Setup, Validation-Split, Benchmark) |
| 03a | `03a_modelling-logreg.ipynb` | Logistische Regression |
| 03b | `03b_modelling-rf.ipynb` | Random Forest |
| 04 | `04_evaluation.ipynb` | **Ergebnis-SSoT** — alle Modelle auf demselben Held-out-Test, Threshold-Tuning, Scoring |
| 04a | `04a_evaluation-baseline.ipynb` | Baseline-Evaluation (exploratorisch) |
| 04b | `04b_evaluation-logreg.ipynb` | LogReg-Deployment-Walkthrough (exploratorisch) |
| 05 | `05_experiment_framework.ipynb` | Engineering-Showcase — Feature-/Model-Catalog + ModelTracker |

## Setup

```bash
uv venv && source .venv/bin/activate
uv pip install -e .            # für pytest/ruff/black: ".[dev]"
```

Projekt-Code und EDA-/Experiment-Helfer liegen im installierbaren Paket `us_used_vehicle_resales`:

```python
from us_used_vehicle_resales.cleaning import clean_data
from us_used_vehicle_resales.features import engineer_features
import us_used_vehicle_resales as wg      # ModelTracker, print_*, save_*, inspect_*
```